# Designing system architectures

**Optional depth track · module 3 of 5**

**Goal:** Write down one architecture decision so that a future reader can tell whether it is still the right one, and check which imports break the boundary you drew.

**Why it matters:** The right architecture is a moving target: what is right for a prototype is not right for the first production system. That only works if the decision recorded the condition under which it expires. Most do not, which is why teams argue about architecture from memory.

Nothing here is graded and nothing in the fifteen sessions depends on it. Work through it when you want the layer underneath.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

In [ ]:
# depth_checks registers this track's checkers. check() and review() are the
# same ones the course uses.
from bootcamp_agent.checks import check, review
import bootcamp_agent.depth_checks  # noqa: F401

## 1. One decision, with an expiry condition

**Context.** Your assistant is synchronous: a question comes in, retrieval and the model call
happen in the request, an answer goes out. That is almost certainly right today. Record it so
that in six months somebody can tell whether it is still right.

**Instructions.**

1. State the decision and the context it was made in.
2. Name one alternative you actually considered, and why not.
3. `reverses_when` is the exercise. The check refuses it without **a number and a unit**,
   because "when it gets slow" is an opinion and nobody can act on it.

In [ ]:
adr = {
    "decision": "Answer requests synchronously in one process. No queue, no worker.",
    "context": "",         # TODO(you): load, latency budget, who runs it
    "alternative": "",     # TODO(you): one you actually considered
    "why_not": "",         # TODO(you): why it lost, today
    "reverses_when": "",   # TODO(you): the measurement that flips this. Number and unit.
}
for k, v in adr.items():
    print(f"{k:16} {v or '(empty)'}")

**Expected output**

```
decision         Answer requests synchronously in one process. No queue, no worker.
context          One cohort, tens of questions an hour, ...
alternative      A task queue with a worker pool and a polling client.
why_not          It adds a broker, a worker deployment ...
reverses_when    p95 request time stays above 2000 ms for 15 minutes ...
✅ d3-e1 passed
```

In [ ]:
check("d3-e1", adr)

## 2. Which import breaks the boundary

**Context.** A boundary you cannot test is a diagram. The rule here: the API talks to the
domain seam, the domain knows nothing about storage or transport, and storage never calls
back up.

**Instructions.**

1. For each import, answer `True` if it breaks that rule and `False` if it is fine.
2. Say it out loud before you answer. Two of these look wrong and are fine.

In [ ]:
verdicts = {
    "api imports agent": False,       # the API calling the domain seam is the point
    "api imports retrieval": None,    # TODO(you)
    "agent imports storage": None,    # TODO(you)
    "storage imports agent": None,    # TODO(you)
    "cli imports agent": None,        # TODO(you)
}
for edge, breaks in verdicts.items():
    print(f"{'breaks' if breaks else 'ok    '}  {edge}")

**Expected output**

```
ok      api imports agent
breaks  api imports retrieval
breaks  agent imports storage
breaks  storage imports agent
ok      cli imports agent
✅ d3-e2 passed
```

In [ ]:
check("d3-e2", verdicts)

## Review

The scorecard for this module. Every ❌ names the exercise and the hint.

In [ ]:
review("d3")